In [ ]:
!pip install transformers datasets seqeval accelerate scikit-learn -q

In [ ]:
import numpy as np
import unicodedata
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModel,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from seqeval.metrics import classification_report
from sklearn.cluster import AgglomerativeClustering

In [ ]:
from google.colab import files

print("Upload BIO tag file (bio_tag.txt):")
uploaded = files.upload()
bio_file = "bio_tag.txt"

print("Upload generic targets file (generic_target.txt):")
uploaded = files.upload()
generic_file = "generic_target.txt"

In [ ]:
generic_targets = []
with open(generic_file, "r", encoding="utf-8") as f:
    for line in f:
        entry = line.strip()
        if entry:
            generic_targets.append(entry)

print(f"Generic targets loaded: {len(generic_targets)}")

In [ ]:
sentences = []
labels = []
current_words = []
current_labels = []

with open(bio_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line == "":
            if current_words:
                sentences.append(current_words)
                labels.append(current_labels)
                current_words = []
                current_labels = []
        else:
            parts = line.split(",")
            if len(parts) == 2:
                word, tag = parts
                current_words.append(word.strip())
                current_labels.append(tag.strip())

if current_words:
    sentences.append(current_words)
    labels.append(current_labels)

print(f"Total sentences loaded: {len(sentences)}")

In [ ]:
VALID_LABELS = {
    "O",
    "B-POLITICIAN", "I-POLITICIAN",
    "B-PARTY",      "I-PARTY",
    "B-POLICY",     "I-POLICY",
    "B-INSTITUTION","I-INSTITUTION",
}

LABEL_CORRECTIONS = {
    "B-POLITTCIAN": "B-POLITICIAN",
    # "I-POLITTCIAN": "I-POLITICIAN",
    # "B-POLITICAIN": "B-POLITICIAN",
    # "I-POLITICAIN": "I-POLITICIAN",
}

for i, sentence_labels in enumerate(labels):
    for j, tag in enumerate(sentence_labels):
        tag = LABEL_CORRECTIONS.get(tag, tag)
        if tag not in VALID_LABELS:
            tag = "O"
        labels[i][j] = tag

In [ ]:
LABEL_LIST = [
    "O",
    "B-POLITICIAN", "I-POLITICIAN",
    "B-PARTY",      "I-PARTY",
    "B-POLICY",     "I-POLICY",
    "B-INSTITUTION","I-INSTITUTION",
]

label2id = {label: idx for idx, label in enumerate(LABEL_LIST)}
id2label = {idx: label for label, idx in label2id.items()}

numeric_labels = [
    [label2id[tag] for tag in sentence_labels]
    for sentence_labels in labels
]

In [ ]:
raw_dataset = Dataset.from_dict({
    "tokens":   sentences,
    "ner_tags": numeric_labels,
})

split = raw_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
test_dataset  = split["test"]

print(f"Train size: {len(train_dataset)} | Test size: {len(test_dataset)}")

In [ ]:
MODEL_CHECKPOINT = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True,
    )

    word_ids  = tokenized_inputs.word_ids()
    label_ids = []
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            label_ids.append(example["ner_tags"][word_idx])
        else:
            label_ids.append(-100)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs

tokenized_train = train_dataset.map(
    tokenize_and_align_labels,
    remove_columns=train_dataset.column_names,
)

tokenized_test = test_dataset.map(
    tokenize_and_align_labels,
    remove_columns=test_dataset.column_names,
)

In [ ]:
ner_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
)

training_args = TrainingArguments(
    output_dir="./ner_model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=ner_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
)

trainer.train()

In [ ]:
# # =========================
# # SAVE MODEL FOR INFERENCE
# # =========================
# SAVE_PATH = "/content/drive/MyDrive/target/final_ner_model"

# # 1. Save model + tokenizer (HuggingFace standard format)
# ner_model.save_pretrained(SAVE_PATH)
# tokenizer.save_pretrained(SAVE_PATH)

# # 2. Save label mappings (VERY IMPORTANT for inference)
# import json

# with open(f"{SAVE_PATH}/label_mappings.json", "w", encoding="utf-8") as f:
#     json.dump({
#         "label2id": label2id,
#         "id2label": id2label
#     }, f, ensure_ascii=False, indent=4)

# # 3. Save generic targets (used in fallback)
# with open(f"{SAVE_PATH}/generic_targets.txt", "w", encoding="utf-8") as f:
#     for g in generic_targets:
#         f.write(g + "\n")

# print(f"✅ Model saved at: {SAVE_PATH}")

In [ ]:
# =========================
# STOPWORDS (VIBHAKTI)
# =========================
STOPWORDS = {
    "को", "का", "की",
    "ले", "लाई",
    "बाट", "देखि",
    "द्वारा",
    "मा", "सँग", "संग",
    "तर्फ", "माथि", "तल",
    "पछि", "अघि",
    "सम्म", "भर",
    "विरुद्ध", "लागि","भित्र"
}

In [ ]:
def remove_stopwords_from_sentence(sentence):
    words = sentence.split()
    filtered_words = [w for w in words if w not in STOPWORDS]
    return " ".join(filtered_words)

In [ ]:
def clean_target(target):
    words = target.split()
    cleaned_words = []

    for w in words:
        original_word = w

        # Only remove suffix if word is longer than suffix
        for sw in STOPWORDS:
            if w.endswith(sw) and len(w) > len(sw):

                # SPECIAL CASE: एमाले (do not damage root)
                if w.startswith("एमाले"):
                    w = "एमाले"
                    break

                # General suffix stripping
                w = w[:-len(sw)]
                break

        cleaned_words.append(w)

    return " ".join(cleaned_words).strip()

In [ ]:
# =========================
# PIPELINE FUNCTIONS
# =========================
def preprocess_target(target):
    target = target.strip()
    target = unicodedata.normalize('NFC', target)
    target = target.replace(" ", "")
    return target


def predict_ner(comment):
    words = comment.split()

    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True
    )

    with torch.no_grad():
        outputs = ner_model(**inputs).logits

    predictions = torch.argmax(outputs, dim=2)[0].numpy()
    word_ids = inputs.word_ids()

    predicted_tags = []
    previous_word_idx = None

    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue

        if word_idx != previous_word_idx:
            predicted_tags.append(id2label[predictions[idx]])

        previous_word_idx = word_idx

    return words, predicted_tags


def extract_targets(words, tags):
    targets = []
    current = []

    for word, tag in zip(words, tags):

        if tag.startswith("B-"):
            if current:
                targets.append(" ".join(current))
                current = []
            current.append(word)

        elif tag.startswith("I-") and current:
            current.append(word)

        else:
            if current:
                targets.append(" ".join(current))
                current = []

    if current:
        targets.append(" ".join(current))

    return targets


# def normalize_targets(targets):
#     if len(targets) <= 1:
#         return targets

#     processed = [preprocess_target(t) for t in targets]
#     embeddings = torch.vstack([get_embedding(t) for t in processed])

#     clustering = AgglomerativeClustering(
#         n_clusters=None,
#         distance_threshold=0.3,
#         metric='cosine',
#         linkage='average'
#     )

#     labels = clustering.fit_predict(embeddings.numpy())

#     cluster_map = {}

#     for i, c in enumerate(labels):
#         cluster_map.setdefault(c, []).append(targets[i])

#     normalized = []

#     for words in cluster_map.values():
#         canonical = max(set(words), key=words.count)
#         normalized.append(canonical)

#     return normalized


def fallback_targets(comment, existing_targets):
    found = []

    for g in generic_targets:
        if g in comment:
            if g not in existing_targets:
                found.append(g)

    return found


# def extract_targets_from_comment(comment):

#     # Step 1: NER
#     words, tags = predict_ner(comment)

#     # Step 2: Extract
#     ner_targets = extract_targets(words, tags)

#     # Step 3: Conditional Fallback (ONLY if no NER targets)
#     if len(ner_targets) == 0:
#         fallback = fallback_targets(comment, ner_targets)
#         final_targets = list(set(fallback))
#     else:
#         final_targets = list(set(ner_targets))

#     return final_targets

def extract_targets_from_comment(comment):

    # Step 1: Clean sentence
    cleaned_sentence = remove_stopwords_from_sentence(comment)

    # Step 2: NER
    words, tags = predict_ner(comment)

    # Step 3: Extract
    ner_targets = extract_targets(words, tags)

    # Step 4: Clean targets
    ner_targets = [clean_target(t) for t in ner_targets if t.strip()]

    # Step 5: Conditional fallback
    if len(ner_targets) == 0:
        fallback = fallback_targets(comment, ner_targets)
        fallback = [clean_target(t) for t in fallback]
        final_targets = list(set(fallback))
    else:
        final_targets = list(set(ner_targets))

    return cleaned_sentence, final_targets

In [ ]:
# =========================
# TEST
# =========================
comments = [
    "ओलीले देश बिगारे सरकार निकम्मा छ",
    "कांग्रेस र एमाले मिलेर देश खाए",
    "प्रधानमन्त्री राम्रो काम गरिरहेका छन्",
    "उपेन्द्र यादवको जीवनी पनि समेट्नुपर्छ",
    "संवैधानिक राजतन्त्र र निर्वाचित प्रधानमन्त्री सन्तुलन हो",
    "गणतन्त्रको विकल्प राजतन्त्र होइन",
    "संसद विघटन र अन्तरिम सरकारको बहस",
    "एमालेले गरेको हो यो काम",
    "त्यो केपी ओलीको काम हो",
    "राम र रावणमा कसलाई रोज्ने",
    "नीतिका पक्षहरू अझै मूल्याङ्कनको चरणमा छन्",
    "देश शान्त कहिले हुन्छ भन्ने प्रश्न गम्भीर छ",
    "महेश बस्नेत केपी ओलीको निकट व्यक्ति हुन्",
    "जेनजी आन्दोलनमा केटाकेटी खुसी हुन्छन् भनेर",
    "राजसंस्था नेपाली जनताले फालेको होइन",
    "ओलीले देश बिगारे",
    "केपी ओली राम्रो नेता हुन्",
    "के पी ओलीको काम प्रशंसनीय छ",
    "ओलि जस्तो नेता फेरि पाइँदैन",
    "ओली सरकार असफल भयो",
    "अविनाश जोशी राम्रो नेता हो।",
    "अमन सिंहले राजनीति क्षेत्रमा राम्रो काम गर्छन्।",
    "रेनु दाहालले भरतपुर महानगरपालिकाको मेयरको रूपमा",
    " पुष्पकमल दाहाल 'प्रचण्ड' नेपालको राजनीतिमा एक अत्यन्तै प्रभावशाली र निर्णायक पात्र हुन्।",
    "केपी ओलीको नेतृत्वमा देशले नयाँ दिशा लिएको छ",
"ओली सरकारको नीति विवादास्पद बनेको छ",
"ओलि जस्तो नेता फेरि भेट्न गाह्रो छ",
"महेश बस्नेत केपी ओलीका निकट सहयोगी हुन्",
"के पी ओलीले संसदमा कडा भाषण दिए",
"ओलीले देशको विकासमा योगदान दिएका छन्",
"केपी ओोलीको निर्णयले धेरै आलोचना पाएको छ",
"ओोली सरकारप्रति जनताको असन्तोष बढ्दै गएको छ",
"केपी ओओलीको शैली फरक र आक्रामक देखिन्छ",
"ओली समर्थकहरू अझै पनि सक्रिय छन्",
"राजनीतिमा ओलीको प्रभाव अझै बलियो छ",
"के पी ओलि नेतृत्वको पार्टीभित्र विवाद छ",
"ओोोलीको नाम सामाजिक सञ्जालमा ट्रेन्ड भयो",
"ओलीले नयाँ नीति घोषणा गरेका छन्",
"जनताले ओलीको कामको मूल्याङ्कन गरिरहेका छन्"
]

for c in comments:
    print("\nComment:", c)
    print("Targets:", extract_targets_from_comment(c))

In [ ]:
import pandas as pd

data = []

for c in comments:
    clean_sent, targets = extract_targets_from_comment(c)

    data.append({
        "original_sentence": c,
        "cleaned_sentence": clean_sent,
        "targets": targets   # list format (BEST)
    })

df = pd.DataFrame(data)

print(df.head())

In [ ]:
# =========================
# EVALUATION + CONFUSION MATRIX
# =========================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

# Get predictions on test set
predictions, labels, _ = trainer.predict(tokenized_test)

preds = np.argmax(predictions, axis=2)

true_labels = []
true_predictions = []

for pred, lab in zip(preds, labels):
    current_preds = []
    current_labels = []

    for p, l in zip(pred, lab):
        if l != -100:  # ignore padding
            current_preds.append(id2label[p])
            current_labels.append(id2label[l])

    true_predictions.append(current_preds)
    true_labels.append(current_labels)

# =========================
# SEQEVAL CLASSIFICATION REPORT
# =========================
print("\n===== SEQEVAL CLASSIFICATION REPORT =====\n")
print(classification_report(true_labels, true_predictions))

print("F1 Score:", f1_score(true_labels, true_predictions))
print("Precision:", precision_score(true_labels, true_predictions))
print("Recall:", recall_score(true_labels, true_predictions))

# =========================
# CONFUSION MATRIX (TOKEN LEVEL)
# =========================
flat_true = []
flat_pred = []

for t_seq, p_seq in zip(true_labels, true_predictions):
    flat_true.extend(t_seq)
    flat_pred.extend(p_seq)

labels_order = LABEL_LIST

cm = confusion_matrix(flat_true, flat_pred, labels=labels_order)

# Plot with BLUE gradient (like your image)
# fig, ax = plt.subplots(figsize=(10, 8))
# Plot with better size
fig, ax = plt.subplots(figsize=(8, 6))  # 🔥 resized (smaller & cleaner)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_order)
disp.plot(
    ax=ax,
    cmap="Blues",              # 🔥 KEY CHANGE
    xticks_rotation=45,
    values_format="d"          # show integers clearly
)

plt.title("NER Confusion Matrix")
plt.tight_layout()

# Save as PNG
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()

print("✅ Confusion matrix saved as confusion_matrix.png")